In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import Markdown, display

In [39]:
orders = pd.read_csv("../data/cleaned/Orders.csv")
payment_methods = pd.read_csv("../data/cleaned/PaymentMethods.csv")

orders["is_canceled"] = orders["status"].eq("canceled")
total_orders = len(orders)
canceled_orders = orders["is_canceled"].sum()
cancellation_rate = canceled_orders / total_orders

print(f"Total orders: {total_orders:,}")
print(f"Canceled orders: {canceled_orders:,}")
print(f"Cancellation rate: {cancellation_rate:.2%}")

Total orders: 408,782
Canceled orders: 148,656
Cancellation rate: 36.37%


In [40]:
orders["is_canceled"] = orders["status"].eq("canceled")

payment_summary = orders.groupby("payment_method_id", dropna=False).agg(
    total_orders=("order_id", "size"),
    canceled_orders=("is_canceled", "sum"),
)

payment_summary["cancellation_rate_pct"] = (
    payment_summary["canceled_orders"] / payment_summary["total_orders"] * 100
)

payment_summary.sort_values("cancellation_rate_pct", ascending=False,)

,total_orders,canceled_orders,cancellation_rate_pct
payment_method_id,,,
15,373,364,97.587131
1,1239,991,79.983858
18,696,544,78.160920
2,16912,12325,72.877247
7,10050,7051,70.159204
16,67912,44834,66.017788
10,344,227,65.988372
6,60278,39030,64.749992
11,12809,6804,53.118901


In [41]:
payment_methods = pd.read_csv("../data/cleaned/PaymentMethods.csv")

payment_report = payment_summary.reset_index().merge(
    payment_methods,
    on="payment_method_id",
    how="left",
    validate="one_to_one",
)

payment_report = payment_report.sort_values("cancellation_rate_pct", ascending=False)
payment_report 


,payment_method_id,total_orders,canceled_orders,cancellation_rate_pct,payment_method_name
14,15,373,364,97.587131,mygateway
0,1,1239,991,79.983858,apg
17,18,696,544,78.160920,ublcreditcard
1,2,16912,12325,72.877247,bankalfalah
6,7,10050,7051,70.159204,easypay_ma
15,16,67912,44834,66.017788,payaxis
9,10,344,227,65.988372,internetbanking
5,6,60278,39030,64.749992,easypay
10,11,12809,6804,53.118901,jazzvoucher
11,12,21613,11438,52.921853,jazzwallet


# 
- the highest rate is mygatewy
- the highest number of cancelations is payaxis 
#

In [42]:
payaxis_orders = orders.loc[orders["payment_method_id"].eq(16)].copy()

payaxis_orders["order_month"] = (
    pd.to_datetime(payaxis_orders["order_date"]).dt.to_period("M")
)

monthly_payaxis = payaxis_orders.groupby("order_month", dropna=False).agg(
    total_orders=("order_id", "size"),
    canceled_orders=("is_canceled", "sum"),
)

monthly_payaxis["cancellation_rate_pct"] = (
    monthly_payaxis["canceled_orders"] / monthly_payaxis["total_orders"] * 100
)

monthly_payaxis.round({"cancellation_rate_pct": 2})

,total_orders,canceled_orders,cancellation_rate_pct
order_month,,,
2016-09,5260,3554,67.57
2016-10,2588,2194,84.78
2016-11,10986,8704,79.23
2016-12,2201,1834,83.33
2017-01,2415,2025,83.85
2017-02,1893,1513,79.93
2017-03,1057,847,80.13
2017-04,436,326,74.77
2017-05,9352,5620,60.09


In [43]:
chart_data = monthly_payaxis.loc[monthly_payaxis.index.notna()].reset_index()
chart_data["order_month"] = chart_data["order_month"].dt.to_timestamp()
chart_data = chart_data.sort_values("order_month")

fig = px.line(
    chart_data,
    x="order_month",
    y="cancellation_rate_pct",
    markers=True,
    title="Payaxis cancellation rate by month",
    hover_data={
        "order_month": "|%Y-%m",
        "total_orders": ":,.0f",
        "canceled_orders": ":,.0f",
        "cancellation_rate_pct": ":.2f",
    },
    labels={
        "order_month": "Order month",
        "total_orders": "Total orders",
        "canceled_orders": "Canceled orders",
        "cancellation_rate_pct": "Cancellation rate (%)",
    },
)

payaxis_rate = payment_summary.loc[16, "cancellation_rate_pct"]
fig.add_hline(y=payaxis_rate, line_dash="dash",
              annotation_text=f"Overall Payaxis rate: {payaxis_rate:.2f}%")
fig.show()

In [44]:
monthly_priority = monthly_payaxis.loc[monthly_payaxis.index.notna()].copy()

monthly_priority["gap_vs_overall_pp"] = (
    monthly_priority["cancellation_rate_pct"] - payaxis_rate
)

monthly_priority.sort_values("canceled_orders", ascending=False).head(5).round(2)

,total_orders,canceled_orders,cancellation_rate_pct,gap_vs_overall_pp
order_month,,,,
2016-11,10986,8704,79.23,13.21
2017-05,9352,5620,60.09,-5.92
2018-03,8108,4346,53.60,-12.42
2016-09,5260,3554,67.57,1.55
2017-10,4142,2796,67.50,1.49


In [45]:
most_cancellations = monthly_priority.loc[
    monthly_priority["canceled_orders"].idxmax()
]

highest_rate = monthly_priority.loc[
    monthly_priority["cancellation_rate_pct"].idxmax()
]

print(f"Month with the most cancellations: {most_cancellations.name} ")
print(f"Month with the highest cancellation rate: {highest_rate.name} ")

Month with the most cancellations: 2016-11 
Month with the highest cancellation rate: 2016-10 
